In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pickle

df = pd.read_csv("data/Sleep_health_and_lifestyle_dataset.csv")
df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None')

print("✅ Data loaded. Shape:", df.shape)
df.head(3)

✅ Data loaded. Shape: (374, 13)


,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,None
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,None
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,None


In [2]:
def get_risk(row):
    pts = 0

    # Sleep scoring
    if row['Sleep Duration'] < 6:             pts += 2
    elif row['Sleep Duration'] < 7:           pts += 1

    # Stress scoring
    if row['Stress Level'] >= 8:              pts += 2
    elif row['Stress Level'] >= 6:            pts += 1

    # Activity scoring
    if row['Physical Activity Level'] < 30:   pts += 2
    elif row['Physical Activity Level'] < 60: pts += 1

    # BMI scoring
    if row['BMI Category'] in ['Obese', 'Overweight']: pts += 1

    # Final label
    if pts >= 4:   return 'High'
    elif pts >= 2: return 'Medium'
    else:          return 'Low'

df['Risk'] = df.apply(get_risk, axis=1)

print("✅ Risk column created!")
print("\nRisk level distribution:")
print(df['Risk'].value_counts())

✅ Risk column created!

Risk level distribution:
Risk
Low       213
High      119
Medium     42
Name: count, dtype: int64


In [3]:
cols_to_drop = ['Person ID', 'Blood Pressure', 'Sleep Disorder']
df.drop(columns=cols_to_drop, inplace=True)

print("✅ Dropped unused columns!")
print("Remaining columns:", df.columns.tolist())
print("New shape:", df.shape)

✅ Dropped unused columns!
Remaining columns: ['Gender', 'Age', 'Occupation', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 'Stress Level', 'BMI Category', 'Heart Rate', 'Daily Steps', 'Risk']
New shape: (374, 11)


In [4]:
le = LabelEncoder()

# Find all text columns except our target Risk
text_cols = df.select_dtypes(include='object').columns.tolist()
text_cols = [c for c in text_cols if c != 'Risk']

print("Text columns to encode:", text_cols)

for col in text_cols:
    df[col] = le.fit_transform(df[col])
    print(f"  ✅ {col} encoded")

print("\nPreview after encoding:")
df.head(3)

Text columns to encode: ['Gender', 'Occupation', 'BMI Category']
  ✅ Gender encoded
  ✅ Occupation encoded
  ✅ BMI Category encoded

Preview after encoding:


,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Risk
0,1,27,9,6.1,6,42,6,3,77,4200,High
1,1,28,1,6.2,6,60,8,0,75,10000,Medium
2,1,28,1,6.2,6,60,8,0,75,10000,Medium


In [5]:
print("Risk before encoding:", df['Risk'].unique())

df['Risk'] = le.fit_transform(df['Risk'])

print("Risk after encoding :", df['Risk'].unique())
print("\nRisk distribution after encoding:")
print(df['Risk'].value_counts())

Risk before encoding: ['High' 'Medium' 'Low']
Risk after encoding : [0 2 1]

Risk distribution after encoding:
Risk
1    213
0    119
2     42
Name: count, dtype: int64


In [6]:
X = df.drop(columns=['Risk'])   # input features — everything except Risk
y = df['Risk']                  # target — only the Risk column

print("✅ Split into features and target!")
print("\nFeatures (X) shape:", X.shape)
print("Target  (y) shape:", y.shape)
print("\nFeature columns:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i}. {col}")

✅ Split into features and target!

Features (X) shape: (374, 10)
Target  (y) shape: (374,)

Feature columns:
  1. Gender
  2. Age
  3. Occupation
  4. Sleep Duration
  5. Quality of Sleep
  6. Physical Activity Level
  7. Stress Level
  8. BMI Category
  9. Heart Rate
  10. Daily Steps


In [7]:
sc = StandardScaler()
X_scaled = sc.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("✅ Scaling and splitting done!")
print(f"\n  Total samples  : {len(X)}")
print(f"  Training set   : {len(X_train)} samples (80%)")
print(f"  Testing set    : {len(X_test)} samples (20%)")
print(f"  Features each  : {X_train.shape[1]}")

✅ Scaling and splitting done!

  Total samples  : 374
  Training set   : 299 samples (80%)
  Testing set    : 75 samples (20%)
  Features each  : 10


In [8]:
import os
os.makedirs("data", exist_ok=True)

# Save train/test data
with open('data/train_test_data.pkl', 'wb') as f:
    pickle.dump((X_train, X_test, y_train, y_test), f)

# Save scaler
with open('data/scaler.pkl', 'wb') as f:
    pickle.dump(sc, f)

# Save feature names
feature_names = X.columns.tolist()
with open('data/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print("✅ All files saved!")
print("\nFiles in data folder:")
for f in os.listdir("data/"):
    size = round(os.path.getsize(f"data/{f}") / 1024, 1)
    print(f"  - {f}  ({size} KB)")

✅ All files saved!

Files in data folder:
  - feature_names.pkl  (0.2 KB)
  - scaler.pkl  (0.9 KB)
  - Sleep_health_and_lifestyle_dataset.csv  (23.6 KB)
  - train_test_data.pkl  (37.5 KB)
